# Session 3: MongoDB for Governance Auditing
## Data Ecosystems and Governance in Organizations

This notebook demonstrates how to use MongoDB aggregation pipelines to audit data for governance issues.

**Learning Objectives:**
- Use aggregation to detect data quality issues
- Find GDPR compliance gaps
- Detect potential bias in credit decisions

## Setup and Connection

In [ ]:
# Install pymongo if needed
# !pip install pymongo

In [ ]:
from pymongo import MongoClient
import pandas as pd
from pprint import pprint

# Connect to MongoDB
# Option 1: Local MongoDB
client = MongoClient('mongodb://localhost:27017/')

# Option 2: MongoDB Atlas (replace with your connection string)
# client = MongoClient('mongodb+srv://<username>:<password>@<cluster>.mongodb.net/')

# Select database and collection
db = client['novacred']
collection = db['credit_applications']

print(f"Connected to database: {db.name}")
print(f"Collection: {collection.name}")
print(f"Document count: {collection.count_documents({})}")

## Quick Data Overview

In [ ]:
# View a sample document to understand the structure
sample = collection.find_one()
pprint(sample)

---
# Part 1: Data Quality Audits

These queries map to the **6 Data Quality Dimensions**:
1. Accuracy
2. Completeness
3. Consistency
4. Timeliness
5. Validity
6. Uniqueness

## Audit Query 1: Find Duplicates (Uniqueness)

**Data Quality Dimension:** Uniqueness  
**Issue:** Same person appearing multiple times in the database

In [ ]:
# Find duplicate SSNs - each person should appear only once!
pipeline_duplicates = [
    {
        "$group": {
            "_id": "$applicant_info.ssn",
            "count": {"$sum": 1},
            "names": {"$push": "$applicant_info.full_name"}
        }
    },
    {
        "$match": {
            "count": {"$gt": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

duplicates = list(collection.aggregate(pipeline_duplicates))

print(f"Found {len(duplicates)} duplicate SSNs:")
for dup in duplicates[:5]:  # Show first 5
    print(f"  SSN: {dup['_id']} - Count: {dup['count']} - Names: {dup['names']}")

### 🔴 Governance Issue
If duplicates exist, this could indicate:
- Data entry errors
- System integration issues
- Potential fraud attempts

**Action Required:** Investigate and deduplicate records

## Audit Query 2: Check Consistency

**Data Quality Dimension:** Consistency  
**Issue:** Same field having different encodings (e.g., "Male" vs "M")

In [ ]:
# How many different gender values exist?
pipeline_gender_consistency = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

gender_values = list(collection.aggregate(pipeline_gender_consistency))

print("Gender value distribution:")
print("Expected: 2 distinct values (Male, Female)")
print(f"Actual: {len(gender_values)} distinct values")
print()
for gv in gender_values:
    print(f"  '{gv['_id']}': {gv['count']} records")

In [ ]:
# Check consistency for other categorical fields
def check_field_consistency(field_path, expected_values=None):
    """Check how many distinct values exist for a field"""
    pipeline = [
        {"$group": {"_id": f"${field_path}", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}
    ]
    results = list(collection.aggregate(pipeline))
    
    print(f"\nField: {field_path}")
    print(f"Distinct values: {len(results)}")
    if expected_values:
        print(f"Expected: {expected_values}")
    for r in results:
        print(f"  '{r['_id']}': {r['count']}")

# Check multiple fields
check_field_consistency("applicant_info.employment_status", ["Employed", "Self-Employed", "Unemployed", "Retired"])
check_field_consistency("decision.loan_approved")

### 🔴 Governance Issue
Inconsistent encoding breaks analytics:
- Gender-based statistics will be wrong if "Male" and "M" are counted separately
- Machine learning models may treat these as different categories

**Action Required:** Standardize encoding before analysis

## Audit Query 3: Find Missing Fields (Completeness)

**Data Quality Dimension:** Completeness  
**GDPR Gap:** Missing consent tracking

In [ ]:
# Count records WITHOUT consent timestamp
pipeline_missing_consent = [
    {
        "$match": {
            "consent_timestamp": {"$exists": False}
        }
    },
    {
        "$count": "missing_consent"
    }
]

result = list(collection.aggregate(pipeline_missing_consent))
missing_consent = result[0]['missing_consent'] if result else 0
total_records = collection.count_documents({})

print(f"Records missing consent timestamp: {missing_consent}")
print(f"Total records: {total_records}")
print(f"Percentage missing: {missing_consent/total_records*100:.1f}%")

In [ ]:
# Check other governance-critical fields
governance_fields = [
    ("consent_timestamp", "GDPR Art. 6 - Lawful basis"),
    ("retention_until", "GDPR Art. 5 - Storage Limitation"),
    ("data_source", "GDPR Art. 14 - Transparency"),
    ("processing_purpose", "GDPR Art. 5 - Purpose Limitation")
]

print("\nGovernance Field Completeness Check:")
print("=" * 60)

for field, gdpr_article in governance_fields:
    pipeline = [
        {"$match": {field: {"$exists": False}}},
        {"$count": "missing"}
    ]
    result = list(collection.aggregate(pipeline))
    missing = result[0]['missing'] if result else 0
    
    status = "✅" if missing == 0 else "🔴"
    print(f"{status} {field}: {missing} missing ({gdpr_article})")

### 🔴 GDPR Compliance Issue
Without these fields, you cannot:
- **consent_timestamp**: Prove lawful basis for processing
- **retention_until**: Implement storage limitation
- **data_source**: Demonstrate transparency
- **processing_purpose**: Enforce purpose limitation

**Action Required:** Add governance metadata to all records

## Audit Query 4: Check Data Validity

**Data Quality Dimension:** Validity  
**Issue:** Values that violate business rules

In [ ]:
# Find records with negative income (invalid!)
pipeline_negative_income = [
    {
        "$match": {
            "financials.annual_income": {"$lt": 0}
        }
    },
    {
        "$count": "negative_income"
    }
]

result = list(collection.aggregate(pipeline_negative_income))
negative_income = result[0]['negative_income'] if result else 0

print(f"Records with negative income: {negative_income}")

In [ ]:
# Find unrealistic debt-to-income ratios (DTI > 1 is suspicious)
pipeline_high_dti = [
    {
        "$match": {
            "financials.debt_to_income": {"$gt": 1}
        }
    },
    {
        "$project": {
            "_id": 1,
            "name": "$applicant_info.full_name",
            "dti": "$financials.debt_to_income"
        }
    },
    {
        "$sort": {"dti": -1}
    },
    {
        "$limit": 10
    }
]

high_dti = list(collection.aggregate(pipeline_high_dti))

print(f"\nRecords with DTI > 1 (unrealistic):")
print("Business rule: DTI should typically be < 0.5")
print()
for record in high_dti:
    print(f"  {record['name']}: DTI = {record['dti']:.2f}")

In [ ]:
# Comprehensive validity checks
validity_rules = [
    ("Negative income", {"financials.annual_income": {"$lt": 0}}),
    ("Zero income", {"financials.annual_income": {"$eq": 0}}),
    ("DTI > 1", {"financials.debt_to_income": {"$gt": 1}}),
    ("Negative credit score", {"financials.credit_score": {"$lt": 0}}),
    ("Credit score > 850", {"financials.credit_score": {"$gt": 850}}),
    ("Negative age", {"applicant_info.age": {"$lt": 0}}),
    ("Age > 120", {"applicant_info.age": {"$gt": 120}}),
]

print("\nValidity Rule Violations:")
print("=" * 50)

for rule_name, match_condition in validity_rules:
    count = collection.count_documents(match_condition)
    status = "✅" if count == 0 else "🔴"
    print(f"{status} {rule_name}: {count} violations")

### 🔴 Governance Issue
Invalid data indicates:
- Missing input validation
- Data integration errors
- Potential data corruption

**Action Required:** Implement validation rules at data entry

---
# Part 2: Bias Detection (AI Act Compliance)

**EU AI Act Requirement:** Credit scoring systems must be tested for bias and discrimination.

## Audit Query 5: Detect Potential Bias

**AI Act Requirement:** Fairness Testing  
**Metric:** Disparate Impact (80% Rule)

In [ ]:
# Approval rate by gender - is there disparate impact?
pipeline_gender_bias = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {"$cond": ["$decision.loan_approved", 1, 0]}
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {"$divide": ["$approved", "$total"]}
        }
    },
    {
        "$sort": {"approval_rate": -1}
    }
]

gender_bias = list(collection.aggregate(pipeline_gender_bias))

print("Approval Rates by Gender:")
print("=" * 50)

for group in gender_bias:
    print(f"{group['_id']}: {group['approval_rate']*100:.1f}% "
          f"({group['approved']}/{group['total']})")

In [ ]:
# Calculate disparate impact ratio
def calculate_disparate_impact(results):
    """Calculate disparate impact ratio (80% rule)"""
    if len(results) < 2:
        return None
    
    rates = {r['_id']: r['approval_rate'] for r in results}
    max_rate = max(rates.values())
    
    print("\nDisparate Impact Analysis (80% Rule):")
    print("=" * 50)
    print(f"Highest approval rate: {max_rate*100:.1f}%")
    print(f"80% threshold: {max_rate*0.8*100:.1f}%")
    print()
    
    for group, rate in rates.items():
        ratio = rate / max_rate
        status = "✅ PASS" if ratio >= 0.8 else "🔴 FAIL - INVESTIGATE"
        print(f"{group}: {rate*100:.1f}% (ratio: {ratio:.2f}) {status}")

calculate_disparate_impact(gender_bias)

In [ ]:
# Check for age bias
pipeline_age_bias = [
    {
        "$bucket": {
            "groupBy": "$applicant_info.age",
            "boundaries": [18, 25, 35, 45, 55, 65, 100],
            "default": "Other",
            "output": {
                "total": {"$sum": 1},
                "approved": {"$sum": {"$cond": ["$decision.loan_approved", 1, 0]}}
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {"$divide": ["$approved", "$total"]}
        }
    }
]

age_bias = list(collection.aggregate(pipeline_age_bias))

print("\nApproval Rates by Age Group:")
print("=" * 50)

age_labels = {
    18: "18-24", 25: "25-34", 35: "35-44", 
    45: "45-54", 55: "55-64", 65: "65+"
}

for group in age_bias:
    label = age_labels.get(group['_id'], str(group['_id']))
    print(f"{label}: {group['approval_rate']*100:.1f}% "
          f"({group['approved']}/{group['total']})")

### 🔴 AI Act Compliance Issue

**The 80% Rule (Four-Fifths Rule):**
- If one group's selection rate is less than 80% of the highest group's rate
- This triggers an "adverse impact" investigation
- Doesn't prove discrimination, but flags it for review

**Action Required:** 
- Document the disparity
- Investigate root causes
- Implement bias mitigation if warranted

---
# Part 3: Governance Summary Report

In [ ]:
# Generate a comprehensive governance audit report
def generate_audit_report():
    print("="*60)
    print("DATA GOVERNANCE AUDIT REPORT")
    print("NovaCred Credit Application Database")
    print("="*60)
    print()
    
    total = collection.count_documents({})
    print(f"Total Records: {total}")
    print()
    
    # Uniqueness
    print("1. UNIQUENESS CHECK")
    dups = len(list(collection.aggregate([
        {"$group": {"_id": "$applicant_info.ssn", "count": {"$sum": 1}}},
        {"$match": {"count": {"$gt": 1}}}
    ])))
    print(f"   Duplicate SSNs found: {dups}")
    print()
    
    # Consistency
    print("2. CONSISTENCY CHECK")
    gender_vals = len(list(collection.aggregate([
        {"$group": {"_id": "$applicant_info.gender"}}
    ])))
    print(f"   Gender encoding values: {gender_vals} (expected: 2)")
    print()
    
    # Completeness
    print("3. COMPLETENESS CHECK (GDPR Fields)")
    for field in ["consent_timestamp", "retention_until", "data_source"]:
        missing = collection.count_documents({field: {"$exists": False}})
        pct = missing/total*100
        status = "✅" if missing == 0 else "🔴"
        print(f"   {status} {field}: {missing} missing ({pct:.1f}%)")
    print()
    
    # Validity
    print("4. VALIDITY CHECK")
    neg_income = collection.count_documents({"financials.annual_income": {"$lt": 0}})
    high_dti = collection.count_documents({"financials.debt_to_income": {"$gt": 1}})
    print(f"   Negative income: {neg_income}")
    print(f"   DTI > 1: {high_dti}")
    print()
    
    # Bias
    print("5. BIAS CHECK (AI Act)")
    gender_results = list(collection.aggregate([
        {"$group": {
            "_id": "$applicant_info.gender",
            "total": {"$sum": 1},
            "approved": {"$sum": {"$cond": ["$decision.loan_approved", 1, 0]}}
        }},
        {"$addFields": {"rate": {"$divide": ["$approved", "$total"]}}}
    ]))
    for g in gender_results:
        print(f"   {g['_id']}: {g['rate']*100:.1f}% approval rate")
    
    print()
    print("="*60)
    print("END OF REPORT")
    print("="*60)

generate_audit_report()

---
# Summary: Query Patterns for Governance

| Query Pattern | Detects | Governance Issue |
|--------------|---------|------------------|
| `$group` + `count > 1` | Duplicates | Uniqueness violation |
| `$group` on field values | Multiple encodings | Consistency problem |
| `$match: {$exists: false}` | Missing data | GDPR gaps |
| `$match` with range conditions | Invalid values | Validity issues |
| `$group` + approval rate | Disparate impact | AI Act fairness |

**Your Project:** Use these exact patterns to audit the credit dataset!